<a href="https://colab.research.google.com/github/JAVERIAADIL/Learning-GPU-infrastructure/blob/module1/kv_cache.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {device}")

Running on: cuda


In [3]:
class kv_cache():
  def __init__(self, d_model, num_of_heads, input, output):
    self.d_model = d_model
    self.num_of_heads = num_of_heads
    self.d_k = d_model // num_of_heads
    self.input = input
    self.output = output
    self.wk = nn.Linear(self.d_model, self.d_model).to(device)
    self.wv = nn.Linear(self.d_model, self.d_model).to(device)
    self.wq = nn.Linear(self.d_model, self.d_model).to(device)
    self.wo = nn.Linear(self.d_model, self.d_model).to(device)
  def calculate_kv(self):
    self.k_cache = []
    self.v_cache = []
    K = self.wk(self.input)
    V = self.wv(self.input)
    seq_len = self.input.shape[0]
    K= K.reshape(seq_len, self.num_of_heads, self.d_k).transpose(0,1)
    V = V.reshape(seq_len, self.num_of_heads, self.d_k).transpose(0,1)
    self.k_cache.append(K)
    self.v_cache.append(V)
    return K, V
  def compute_Q(self, K, V):
    for step in range(self.output):
      self.last_token = self.input[-1:]
      Q = self.wq(self.last_token)  #here last_token @ wq.T happens so qC = wqR and last_token(1, d_model) so we take wq(d_model, here also d_model or d_k if we use heads as we later have to multiply it with K ) so we get Q (1, d_model or d_k)
      Q = Q.reshape(1, self.num_of_heads, self.d_k)
      Q = Q.transpose(0,1)
      scores = Q @ K.transpose(-2, -1) / torch.sqrt((torch.tensor(self.d_k, dtype=torch.float32)))
      attention = torch.softmax(scores, dim=-1)
      output_token = attention @ V
      print(output_token.shape) #(2, 1, 4) as 4 is d_k
      output_token = output_token.transpose(0,1) #(1,2,4)
      output_token = output_token.reshape(1, self.d_model) #d_model = 8
      output_token = self.wo(output_token)
      print(f"output : {output_token}")
      new_K = self.wk(output_token) # wk(d_model, d_model)
      new_V = self.wv(output_token)
      #before appending it to kv cache we have to reshape it with heads
      new_K = new_K.reshape(1, self.num_of_heads, self.d_k).transpose(0,1)
      new_V = new_V.reshape(1, self.num_of_heads, self.d_k).transpose(0,1)
      self.k_cache.append(new_K)
      self.v_cache.append(new_V)
      self.input = torch.cat([self.input, output_token], dim=0) #we concatenate output with input to generate new token
      pass
    self.final_input = self.input
    return self.k_cache, self.v_cache, self.final_input
  def output_function(self):
    K, V = self.calculate_kv()
    k_cache, v_cache, input = self.compute_Q(K, V)
    return k_cache, v_cache, input




In [10]:
input = torch.randn(4,8).to(device) #means we have 4 input seq_len and each have 8 dimension d_model
d_model = 8
num_of_heads = 2
output = 5 #suppose we need 5 token as output to generate by model
kv = kv_cache(d_model, num_of_heads, input, output)
k_cache, v_cache, final_input = kv.output_function()
print(f"k_cache: {k_cache}, v_cache :{v_cache}")
print(f"k_cache of 1 token Shape: {k_cache[0].shape}, v_cache Shape of 1 token:{v_cache[0].shape}")
print(f"k_cache length: {len(k_cache)}, v_cache length:{len(v_cache)}")
print(f"complete Tokens : {final_input}, shape :{ final_input.shape}")

torch.Size([2, 1, 4])
output : tensor([[ 0.0282,  0.6924, -0.3800, -0.5771, -0.1589,  0.0038, -0.0450,  0.0293]],
       device='cuda:0', grad_fn=<ViewBackward0>)
torch.Size([2, 1, 4])
output : tensor([[ 0.0407,  0.6238, -0.4893, -0.6122, -0.1996, -0.0530, -0.0915, -0.0106]],
       device='cuda:0', grad_fn=<ViewBackward0>)
torch.Size([2, 1, 4])
output : tensor([[ 0.0411,  0.6278, -0.4877, -0.6112, -0.2044, -0.0553, -0.0927, -0.0141]],
       device='cuda:0', grad_fn=<ViewBackward0>)
torch.Size([2, 1, 4])
output : tensor([[ 0.0411,  0.6279, -0.4873, -0.6111, -0.2044, -0.0552, -0.0926, -0.0140]],
       device='cuda:0', grad_fn=<ViewBackward0>)
torch.Size([2, 1, 4])
output : tensor([[ 0.0411,  0.6279, -0.4873, -0.6111, -0.2044, -0.0552, -0.0925, -0.0140]],
       device='cuda:0', grad_fn=<ViewBackward0>)
k_cache: [tensor([[[ 0.7223, -1.3544, -0.4038, -0.8614],
         [ 0.1539,  0.3872, -1.1441, -0.2002],
         [ 0.0722,  0.6451, -0.1167,  0.5243],
         [-0.3913,  0.4519, -0.143

If you have num_heads=8 and d_k=64, what is the shape of the full KV cache after generating 200 tokens on top of a 50-token prompt?
Ans: d_k = d_model// num_of_heads so d_model = 512 means input is (50, 512) so 50+200 = 250 so k_cache (250, 512) v_cache (250, 512) and since there are 8 heads and kv cache store split-head version not the pre-split version so k_cache(8,250,512) and v_cache(8,250,512)